# Sembolsüz x86-64 fonksiyon adlandırma — QLoRA

Bu notebook, ölçek verisini Qwen2.5-Coder üzerinde 4-bit QLoRA ile yalnız `ad` hedefi için eğitir. Kayıp yalnız assistant yanıtında hesaplanır; sonunda tohumlu en çok 2.000 test örneği ve sabit 115 örneklik eski küme ayrı ölçülür.

Colab'da önce **Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU** seçin. Varsayılan model A100 için 7B'dir; L4'te 3B karşılığı otomatik seçilebilir.

## 1. Sabit sürümlü bağımlılıklar

In [ ]:
%pip install -q \
  "transformers==4.48.3" \
  "peft==0.14.0" \
  "trl==0.15.2" \
  "bitsandbytes>=0.46.1" \
  "accelerate==1.3.0" \
  "datasets==3.2.0"

## 2. Ayarlar ve GPU'ya göre model seçimi

In [ ]:
from pathlib import Path
import json
import math
import random
import re
import shutil
import zipfile

import torch

# "yukleme": veri.zip dosyasını Colab'ın sol dosya panelinden /content'e yükleyin.
# "drive": DRIVE_ZIP yolundaki arşivi kullanın. VERI_KLASORU arşiv içindeki alt klasördür.
VERI_KAYNAGI = "yukleme"                 # "yukleme" veya "drive"
YUKLENEN_ZIP = "/content/veri.zip"
DRIVE_ZIP = "/content/drive/MyDrive/asm-adlandirma/veri.zip"
VERI_KLASORU = ""                         # zip_hazirla.sh için arşiv kökü

MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
BELLEGE_GORE_MODEL_SEC = True
L4_MODELI = "Qwen/Qwen2.5-Coder-3B-Instruct"

MAX_UZUNLUK = 3072
ETKIN_BATCH = 16
MIKRO_BATCH = 2
EPOCH = 2.0
MAX_ORNEK = 41000                          # 0 = hepsi; v4 için ~6 saatlik A100 önerisi
MAX_ADIM = 0                               # 0 = EPOCH tamamlanana kadar
VALID_TAVAN = 500
TEST_TAVAN = 2000
ZAMAN_BUTCESI_SAAT = 6.0
A100_TOKEN_SN = 3500
TOHUM = 7

if not torch.cuda.is_available():
    raise RuntimeError("GPU bulunamadı; Colab çalışma zamanını GPU olarak değiştirin.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("Bu notebook bf16 destekli A100 veya L4 bekliyor.")
if ETKIN_BATCH % MIKRO_BATCH:
    raise ValueError("ETKIN_BATCH, MIKRO_BATCH değerine tam bölünmeli.")

gpu = torch.cuda.get_device_properties(0)
bellek_gb = gpu.total_memory / 1024**3
SECILEN_MODEL = MODEL if not (BELLEGE_GORE_MODEL_SEC and bellek_gb < 35) else L4_MODELI
BIRIKIM_ADIMI = ETKIN_BATCH // MIKRO_BATCH
model_kisa = SECILEN_MODEL.rsplit("/", 1)[-1]

print(f"GPU: {gpu.name} ({bellek_gb:.1f} GB)")
print(f"Model: {SECILEN_MODEL}")
print(f"Etkin batch: {MIKRO_BATCH} × {BIRIKIM_ADIMI} = {ETKIN_BATCH}")

## 3. Veriyi yükleme veya Drive'dan okuma

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
if VERI_KAYNAGI == "drive":
    zip_yolu = Path(DRIVE_ZIP)
elif VERI_KAYNAGI == "yukleme":
    zip_yolu = Path(YUKLENEN_ZIP)
else:
    raise ValueError("VERI_KAYNAGI yalnız 'yukleme' veya 'drive' olabilir.")
VERI_URL = "https://github.com/krxi/asm-adlandirma-veri/releases/download/v4/veri-olcek-v4.zip"
if not zip_yolu.is_file() and VERI_KAYNAGI == "yukleme":
    import urllib.request
    print(f"{zip_yolu} yok; indiriliyor: {VERI_URL}")
    urllib.request.urlretrieve(VERI_URL, zip_yolu)
if not zip_yolu.is_file():
    raise FileNotFoundError(f"Arşiv bulunamadı: {zip_yolu}")

calisma_dizini = Path("/content/asm-adlandirma")
acma_dizini = calisma_dizini / "veri-acilmis"
acma_dizini.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_yolu) as z:
    z.extractall(acma_dizini)

VERI_DIZINI = acma_dizini / VERI_KLASORU
zorunlu = ("train", "valid", "test", "eval115")
if not all((VERI_DIZINI / f"{ad}.jsonl").is_file() for ad in zorunlu):
    raise FileNotFoundError(f"{VERI_DIZINI} içinde {zorunlu} JSONL dosyaları bulunamadı.")

for ad in zorunlu:
    yol = VERI_DIZINI / f"{ad}.jsonl"
    with yol.open(encoding="utf-8") as f:
        adet = sum(1 for satir in f if satir.strip())
    print(f"{ad:>7}: {adet:>6} satır  ({yol})")

## 4. 4-bit model ve tüm lineer katmanlarda LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

set_seed(TOHUM)
tokenizer = AutoTokenizer.from_pretrained(SECILEN_MODEL, use_fast=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

niceleme = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    SECILEN_MODEL,
    quantization_config=niceleme,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    attn_implementation="sdpa",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
model = get_peft_model(
    model,
    LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules="all-linear",
    ),
)
model.print_trainable_parameters()

## 5. Sohbet şablonu ve yalnız assistant tokenlarında kayıp

Qwen sohbet şablonuyla kullanıcı istemi ve tam konuşma ayrı kodlanır. İsteme karşılık gelen etiketler `-100` yapıldığı için sistem ve kullanıcı tokenları kayba katılmaz.

In [ ]:
from datasets import load_dataset

ham_veri = load_dataset(
    "json",
    data_files={
        "train": str(VERI_DIZINI / "train.jsonl"),
        "validation": str(VERI_DIZINI / "valid.jsonl"),
    },
)
if MAX_ORNEK and len(ham_veri["train"]) > MAX_ORNEK:
    ham_veri["train"] = ham_veri["train"].shuffle(seed=TOHUM).select(range(MAX_ORNEK))
ham_veri["validation"] = ham_veri["validation"].shuffle(seed=TOHUM).select(
    range(min(VALID_TAVAN, len(ham_veri["validation"])))
)

def kodla(ornek):
    mesajlar = ornek["messages"]
    if [m["role"] for m in mesajlar] != ["system", "user", "assistant"]:
        raise ValueError("Beklenen mesaj sırası: system, user, assistant")
    hedef = json.loads(mesajlar[2]["content"])
    if set(hedef) != {"ad"}:
        raise ValueError(f"Hedef yalnız ad içermeli: {hedef}")

    istem = tokenizer.apply_chat_template(
        mesajlar[:2], tokenize=True, add_generation_prompt=True
    )
    tumu = tokenizer.apply_chat_template(
        mesajlar, tokenize=True, add_generation_prompt=False
    )
    if tumu[:len(istem)] != istem:
        raise ValueError("Sohbet şablonunda assistant başlangıcı belirlenemedi.")

    tumu = tumu[:MAX_UZUNLUK]
    etiketler = [-100] * min(len(istem), len(tumu)) + tumu[len(istem):]
    if not any(x != -100 for x in etiketler):
        raise ValueError("Yanıt MAX_UZUNLUK dışında kaldı; veri hazırlama tavanını düşürün.")
    return {
        "input_ids": tumu,
        "attention_mask": [1] * len(tumu),
        "labels": etiketler,
        "uzunluk": len(tumu),
    }

tokenli_veri = ham_veri.map(
    kodla,
    remove_columns=ham_veri["train"].column_names,
    desc="Qwen sohbet şablonu uygulanıyor",
)

egitim_uzunluklari = tokenli_veri["train"]["uzunluk"]
for bolum in ("train", "validation"):
    uzunluklar = tokenli_veri[bolum]["uzunluk"]
    p95 = sorted(uzunluklar)[math.ceil(0.95 * len(uzunluklar)) - 1]
    print(
        f"{bolum:>10}: {len(uzunluklar)} örnek; "
        f"ortalama {sum(uzunluklar) / len(uzunluklar):.0f}, p95 {p95}, en uzun {max(uzunluklar)} token"
    )

ortalama_token = sum(egitim_uzunluklari) / len(egitim_uzunluklari)
adim_epoch = math.ceil(len(egitim_uzunluklari) / ETKIN_BATCH)
dogal_adim = math.ceil(adim_epoch * EPOCH)
planlanan_adim = MAX_ADIM if MAX_ADIM > 0 else dogal_adim
planlanan_token = round(planlanan_adim * ETKIN_BATCH * ortalama_token)
tahmini_saat = planlanan_token / A100_TOKEN_SN / 3600
butce_token = ZAMAN_BUTCESI_SAAT * 3600 * A100_TOKEN_SN
onerilen_max_ornek = min(
    len(egitim_uzunluklari),
    max(1, math.floor(butce_token / (ortalama_token * EPOCH))),
)
print(f"Plan: {planlanan_adim:,} adım, yaklaşık {planlanan_token:,} token, A100'de {tahmini_saat:.2f} saat")
if tahmini_saat > ZAMAN_BUTCESI_SAAT:
    print(f"{ZAMAN_BUTCESI_SAAT:g} saat için önerilen MAX_ORNEK ≈ {onerilen_max_ornek:,}")
elif MAX_ORNEK:
    print(f"Seçili MAX_ORNEK={MAX_ORNEK:,}, {ZAMAN_BUTCESI_SAAT:g} saatlik bütçeye sığıyor.")
else:
    print(f"Tam veri {ZAMAN_BUTCESI_SAAT:g} saatlik bütçeye sığıyor; MAX_ORNEK=0 kalabilir.")

tokenli_veri = tokenli_veri.remove_columns("uzunluk")

## 6. Süre/adım sınırlı, Drive checkpoint'li eğitim

In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

DENETIM_DIZINI = Path("/content/drive/MyDrive/asm-adlandirma") / f"denetim-{model_kisa}"
DENETIM_DIZINI.mkdir(parents=True, exist_ok=True)
egitim_ayari = TrainingArguments(
    output_dir=str(DENETIM_DIZINI),
    num_train_epochs=EPOCH,
    max_steps=MAX_ADIM if MAX_ADIM > 0 else -1,
    per_device_train_batch_size=MIKRO_BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=BIRIKIM_ADIMI,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="paged_adamw_8bit",
    bf16=True,
    tf32=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=0.3,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    logging_steps=10,
    report_to="none",
    seed=TOHUM,
    data_seed=TOHUM,
    group_by_length=True,
    remove_unused_columns=False,
)

veri_toplayici = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
    return_tensors="pt",
)
egitici = Trainer(
    model=model,
    args=egitim_ayari,
    train_dataset=tokenli_veri["train"],
    eval_dataset=tokenli_veri["validation"],
    data_collator=veri_toplayici,
)
son_checkpoint = get_last_checkpoint(str(DENETIM_DIZINI))
print(f"Devam checkpoint'i: {son_checkpoint or 'yok; sıfırdan başlanacak'}")
egitim_sonucu = egitici.train(resume_from_checkpoint=son_checkpoint)
egitici.evaluate()
egitim_sonucu

Devam checkpoint'i: yok; sıfırdan başlanacak


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss


## 7. Adaptörü Drive'a kaydetme ve zip olarak indirme

In [ ]:
from google.colab import files

ADAPTOR_DIZINI = Path("/content/drive/MyDrive/asm-adlandirma") / f"adaptor-{model_kisa}"
ADAPTOR_DIZINI.parent.mkdir(parents=True, exist_ok=True)

egitici.save_model(str(ADAPTOR_DIZINI))
tokenizer.save_pretrained(str(ADAPTOR_DIZINI))

zip_koku = Path("/content") / ADAPTOR_DIZINI.name
zip_yolu = Path(shutil.make_archive(
    str(zip_koku),
    "zip",
    root_dir=ADAPTOR_DIZINI.parent,
    base_dir=ADAPTOR_DIZINI.name,
))
print(f"Drive: {ADAPTOR_DIZINI}")
print(f"İndirilecek arşiv: {zip_yolu} ({zip_yolu.stat().st_size / 1024**2:.0f} MB)")
files.download(str(zip_yolu))

## 8. Test ve eval115 ölçümü

Üretim greedy'dir (`do_sample=False`) ve yalnız ad JSON'u için en çok 48 yeni token kullanır. Test kümesi tohumlu olarak en çok 2.000 örneğe indirilir; 115'lik eski küme bütünüyle ve ayrı raporlanır.

In [ ]:
import gc
from tqdm.auto import tqdm

# lora/olc.py, taban.py içindeki bu iki fonksiyonu kullanır; tanım birebir kopyadır.
def kelimeler(ad: str) -> list[str]:
    ad = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", ad)
    return [k for k in re.split(r"[_\W]+", ad.lower()) if k]


def f1(tahmin: str, gercek: str) -> float:
    t, g = kelimeler(tahmin), kelimeler(gercek)
    ortak = len(set(t) & set(g))
    if not ortak:
        return 0.0
    p, r = ortak / len(set(t)), ortak / len(set(g))
    return 2 * p * r / (p + r)


def oku_ve_ornekle(yol: Path, tavan: int) -> list[dict]:
    with yol.open(encoding="utf-8") as f:
        satirlar = [json.loads(satir) for satir in f if satir.strip()]
    if tavan and len(satirlar) > tavan:
        satirlar = random.Random(TOHUM).sample(satirlar, tavan)
    return satirlar


def olc(dosya: str, etiket: str, tavan: int) -> Path:
    satirlar = oku_ve_ornekle(VERI_DIZINI / dosya, tavan)
    sonuclar = []
    for kayit in tqdm(satirlar, desc=f"Greedy {etiket} üretimi"):
        mesajlar = kayit["messages"][:2]
        girdi = tokenizer.apply_chat_template(
            mesajlar, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        if girdi.shape[1] > MAX_UZUNLUK:
            raise ValueError(f"Girdi {MAX_UZUNLUK} tokenı aşıyor: {kayit.get('id')}")
        with torch.inference_mode():
            uretilen = model.generate(
                input_ids=girdi,
                attention_mask=torch.ones_like(girdi),
                max_new_tokens=48,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        yeni_tokenlar = uretilen[0, girdi.shape[1]:]
        metin = tokenizer.decode(yeni_tokenlar, skip_special_tokens=True)
        eslesmeler = re.findall(r"\{[^{}]*\}", metin)
        try:
            cevap = json.loads(eslesmeler[-1]) if eslesmeler else {"ad": metin.strip()[:60]}
        except json.JSONDecodeError:
            cevap = {}
        tahmin = str(cevap.get("ad", ""))
        gercek = str(json.loads(kayit["messages"][2]["content"])["ad"])
        skor = f1(tahmin, gercek)
        sonuclar.append({
            "id": kayit.get("id", ""),
            "gercek": gercek,
            "tahmin": tahmin,
            "aciklama": "",
            "ogretmen": "",
            "f1": round(skor, 3),
            "opt": kayit.get("opt", ""),
            "proje": kayit.get("proje", ""),
            "token": int(girdi.shape[1] + yeni_tokenlar.shape[0]),
        })

    sonuc_yolu = Path("/content") / f"sonuc-{etiket}-{model_kisa}-lora.jsonl"
    with sonuc_yolu.open("w", encoding="utf-8") as f:
        for sonuc in sonuclar:
            f.write(json.dumps(sonuc, ensure_ascii=False) + "\n")
    ortalama = sum(r["f1"] for r in sonuclar) / len(sonuclar)
    tam = sum(r["f1"] == 1 for r in sonuclar)
    print(f"{etiket}: ortalama F1 {ortalama:.3f} (n={len(sonuclar)}, tam isabet {tam})")
    for opt in ("-O0", "-O1", "-O2", "-O3", "-Os"):
        alt = [r["f1"] for r in sonuclar if r["opt"] == opt]
        if alt:
            print(f"  {opt}: {sum(alt) / len(alt):.3f} (n={len(alt)})")
    print(f"ayrıntı → {sonuc_yolu}")
    return sonuc_yolu


del egitici
gc.collect()
torch.cuda.empty_cache()
model.config.use_cache = True
model.eval()

sonuc_yollari = [
    olc("test.jsonl", "test", TEST_TAVAN),
    olc("eval115.jsonl", "eval115", 0),
]
for sonuc_yolu in sonuc_yollari:
    files.download(str(sonuc_yolu))